In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [6]:
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.mappers import JordanWignerMapper, ParityMapper, BravyiKitaevMapper
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper, BravyiKitaevMapper, ParityMapper
from qiskit_nature.units import DistanceUnit

def H2_qubits(bond_length, mapper_type):
    
    atom_string = f"H 0 0 0; H 0 0 {bond_length}"

    driver = PySCFDriver(
        atom=atom_string,
        basis="sto3g",
        charge=0,
        spin=0,
        unit=DistanceUnit.ANGSTROM,
    )

    es_problem = driver.run()
    second_q_op = es_problem.hamiltonian.second_q_op()

    # 1. Tentukan mapper berdasarkan input 'mapper_type'
    if mapper_type.lower() == "jordan_wigner":
        mapper = JordanWignerMapper()
    elif mapper_type.lower() == "bravyi_kitaev":
        mapper = BravyiKitaevMapper()
    elif mapper_type.lower() == "parity":
        mapper = ParityMapper(num_particles=es_problem.num_particles)
    else:
        raise ValueError(f"Mapper type '{mapper_type}' tidak dikenal. Gunakan 'jordan_wigner', 'bravyi_kitaev', atau 'parity'.")

    # 2. Lakukan mapping dari Second Quantization ke Qubit Operator
    hamiltonian = mapper.map(second_q_op)
    
    return hamiltonian.to_list()

bond_lengths = [0.25, 0.45, 0.65, 0.725, 0.80, 1.50, 2.50, 3.00]
data_jw = {}
for r in bond_lengths:
    data_jw[float(r)] = H2_qubits(r, 'jordan_wigner')
    
data_bk = {}
for r in bond_lengths:
    data_bk[float(r)] = H2_qubits(r, 'bravyi_kitaev')


data_par = {}
for r in bond_lengths:
    data_par[float(r)] = H2_qubits(r, 'parity')

In [10]:
dict_jw = [pauli for pauli, coeff in data_jw[0.45]]
print(dict_jw)

['IIII', 'IIIZ', 'IIZI', 'IIZZ', 'IZII', 'IZIZ', 'ZIII', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII']


In [ ]:
dict_bk = [pauli for pauli, coeff in data_bk[0.25]]
print(dict_bk)

['IIII', 'IIIZ', 'IIZZ', 'IIZI', 'IZII', 'IZIZ', 'ZZZI', 'ZZZZ', 'ZXIX', 'IXZX', 'ZXZX', 'IXIX', 'IZZZ', 'ZZIZ', 'ZIZI']


In [26]:
dict_par = [pauli for pauli, coeff in data_par[0.25]]
print(dict_par)

['II', 'IZ', 'ZI', 'ZZ', 'XX']
